# 🚗 QNX Global E-Hack 2026: Production ML Training Pipeline
## 🚦 10-Class Autonomous Vehicle Road Sign Recognition (`det10`)

**Target Deployment:** BlackBerry QNX Neutrino RTOS 8.0 on Raspberry Pi 4 (ARM Cortex-A72)

### 🎯 The 10 Target Road Sign Classes (`det10`):
1. `stop` - Mandatory Halt
2. `no_entry` - Prohibited Access
3. `speed_limit_30` - Speed Limit 30 km/h
4. `school_ahead` - School Zone Advisory (25 km/h)
5. `pedestrian_crossing` - Zebra / Pedestrian Crosswalk
6. `slippery_road` - Low Traction Advisory
7. `right_hand_curve` - Sharp Right Bend Ahead
8. `give_way` - Yield Priority
9. `no_parking` - Continuous Movement
10. `narrow_road_ahead` - Road Constriction

---

In [ ]:
# [Cell 1]: Verify GPU Hardware Environment (NVIDIA T4 / A100 / V100)
!nvidia-smi
!pip install -q ultralytics onnx onnxruntime onnxslim albumentations

In [ ]:
# [Cell 2]: Mount Google Drive / Cloud Storage for 200GB+ Ingestion
from google.colab import drive
drive.mount('/content/drive')

# Or clone repository directly
!git clone https://github.com/vuppurohith10-lab/QNX_INDIA_HACKATHON_HYD_2026_QNXHACK-HYD-019.git /content/qnx_repo

In [ ]:
# [Cell 3]: Create det10 Dataset Configuration YAML
import yaml

dataset_config = {
    'path': '/content/dataset_det10',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 10,
    'names': {
        0: 'stop',
        1: 'no_entry',
        2: 'speed_limit_30',
        3: 'school_ahead',
        4: 'pedestrian_crossing',
        5: 'slippery_road',
        6: 'right_hand_curve',
        7: 'give_way',
        8: 'no_parking',
        9: 'narrow_road_ahead'
    }
}

with open('/content/det10.yaml', 'w') as f:
    yaml.dump(dataset_config, f)
print('✅ /content/det10.yaml created successfully!')

In [ ]:
# [Cell 4]: Execute High-Certainty Production Training (cls=2.5, cos_lr=True, fliplr=0.0)
from ultralytics import YOLO

# Load baseline architecture (YOLOv8s / YOLOv11s)
model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/det10.yaml',
    epochs=50,
    batch=32,
    imgsz=384,             # Optimal resolution balancing edge latency & small sign detection
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,           # Cosine learning rate scheduling for deep global convergence
    weight_decay=0.0005,
    warmup_epochs=3,
    cls=2.5,               # 5x higher classification penalty (sharp softmax probability > 95%)
    box=7.5,               # Strict bounding box regression
    dfl=1.5,
    # Augmentations designed for traffic signs:
    augment=True,
    fliplr=0.0,            # CRITICAL: Preserve directional signage semantics (Right Turn vs Left Turn!)
    flipud=0.0,
    degrees=10.0,          # Slight vehicle roll simulation
    scale=0.25,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    mosaic=0.7,
    close_mosaic=10,
    project='/content/runs_production',
    name='det10_prod',
    exist_ok=True,
    verbose=True
)

In [ ]:
# [Cell 5]: Validate Model Performance & Class-wise mAP Metrics
metrics = model.val(data='/content/det10.yaml', imgsz=384)
print('=' * 60)
print(f'Overall mAP@0.50     : {metrics.box.map50:.4f}')
print(f'Overall mAP@0.50:0.95: {metrics.box.map:.4f}')
print(f'Mean Precision       : {metrics.box.mp:.4f}')
print(f'Mean Recall          : {metrics.box.mr:.4f}')
print('=' * 60)

In [ ]:
# [Cell 6]: Multi-Format Edge Deployment Export (ONNX + TFLite INT8 for QNX RTOS)
import shutil
import os

best_weights = '/content/runs_production/det10_prod/weights/best.pt'
trained_model = YOLO(best_weights)

# 1. Export ONNX (for OpenCV DNN CPU inference on QNX)
onnx_path = trained_model.export(format='onnx', imgsz=320, simplify=True)
print('✅ Exported ONNX:', onnx_path)

# 2. Export TFLite (for Raspberry Pi 4 CPU acceleration)
tflite_path = trained_model.export(format='tflite', imgsz=320)
print('✅ Exported TFLite:', tflite_path)

# 3. Package weights for vehicle transfer
!mkdir -p /content/qnx_delivery/models
!cp /content/runs_production/det10_prod/weights/best.pt /content/qnx_delivery/models/best.pt
!cp {onnx_path} /content/qnx_delivery/models/best.onnx
!cp {tflite_path} /content/qnx_delivery/models/best.tflite

with open('/content/qnx_delivery/models/labels.txt', 'w') as f:
    for i in range(10):
        f.write(f"{dataset_config['names'][i]}\n")

!cd /content/qnx_delivery && zip -r /content/qnx_det10_production_model.zip models/

# Trigger automatic download to user laptop
from google.colab import files
files.download('/content/qnx_det10_production_model.zip')
print('🎉 Production model package downloaded to your computer!')